# 02 — Evaluation: standard + **region-stratified** protocol

Runs on the LOL-v1 **test-15** (never used for tuning), config `configs/ablations/no_illum_prior.yaml`:

| Level | What | Metrics |
|---|---|---|
| 1. Standard | test-15 | PSNR / SSIM / LPIPS |
| 2. **Region-stratified** | dark/mid/bright GT-luminance terciles | PSNR + ΔE2000 **per stratum** |
| 3. Naturalness | LOL-v1 | LOE |
| 4. Cross-dataset (optional) | LIME / NPE / DICM | NIQE / BRISQUE / PI |

**Prerequisites:** the matching `01` kernel has run — its output (checkpoints) is
imported automatically via `kernel_sources`, so this works in a fresh session.

## 1. Environment (clone + install — same as 00, no secrets needed)

`01`'s output (checkpoints) is auto-imported as an input via `kernel_sources`
in `kernel-metadata.json`, so this kernel can run in a fresh session.

In [ ]:
REPO_URL = "https://github.com/pranjalkole/selective-gating.git"   # public repo
BRANCH   = "master"

import os, subprocess, sys, shutil
from pathlib import Path

repo = Path("/kaggle/working/selective-gating")
if repo.exists():
    if subprocess.run(["git", "-C", str(repo), "pull", REPO_URL, BRANCH]).returncode != 0:
        print("pull failed -> fresh clone")
        shutil.rmtree(repo)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)

print("repo at:", repo)

sys.path.insert(0, str(repo / "src"))   # PYTHONPATH only affects child processes, not this kernel
os.environ["PYTHONPATH"] = f"{repo / 'src'}:{os.environ.get('PYTHONPATH', '')}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)

import torch
print("cuda:", torch.cuda.is_available())

## 2. Locate config + checkpoint (searches /kaggle/working and imported kernel outputs)

In [ ]:
CONFIG = 'configs/ablations/no_illum_prior.yaml'
RUN_NAME = Path(CONFIG).stem

candidates = sorted(set(Path("/kaggle/working").rglob("best.pt"))
                    | set(Path("/kaggle/input").rglob("best.pt")))
matches = [p for p in candidates if RUN_NAME in str(p)]
if matches:
    CKPT = matches[0]
else:
    # run-name stem can differ slightly from the yaml's internal name (e.g.
    # sweep_illum_003 vs ablation_sweep_illum003) — only fall back to a
    # lone candidate, never silently pick among several.
    assert len(candidates) == 1, "ambiguous checkpoint selection: " + repr(candidates)
    CKPT = candidates[0]
assert CKPT is not None, "no best.pt found — run the matching 01 kernel first"
print("using checkpoint:", CKPT)

OUT = repo / "results" / RUN_NAME / "test"

## 3. Run standard + stratified evaluation on the test-15

In [ ]:
cmd = [sys.executable, str(repo / "scripts" / "evaluate.py"),
       "--config", str(repo / CONFIG), "--ckpt", str(CKPT),
       "--split", "test", "--out", str(OUT)]
# dataset root is auto-detected from the attached data; override with:
# cmd += ["--set", "data.root=/kaggle/input/<your-lol-dataset>/<subdir>"]

print(" ".join(cmd))
ret = subprocess.run(cmd, cwd=repo).returncode
assert ret == 0, f"evaluation failed (exit {ret})"

## 4. Headline table: per-stratum fidelity

In [ ]:
import csv, json
from pathlib import Path

summary = list(csv.DictReader(open(OUT / "stratified_summary.csv")))
print(f"{'stratum':>8} | {'PSNR (dB)':>10} | {'ΔE2000':>8} | {'images':>7}")
print("-" * 46)
for r in summary:
    print(f"{r['stratum']:>8} | {float(r['psnr']):>10.3f} | {float(r['delta_e2000']):>8.3f} | {r['n_images']:>7}")

metrics = json.loads((OUT / "metrics.json").read_text())
print("\nstandard:", {k: round(v, 4) for k, v in metrics["standard"].items()})

## 5. Visual check: input / output / gate map side by side

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

preds = sorted((OUT / "predictions").glob("*.png"))
print(len(preds), "enhanced images")

def find_pair(name, root_lol=None):
    from selgate.paths import find_lol_root
    from selgate.data import detect_layout
    root = find_lol_root(); layout = detect_layout(root)
    gt = root / layout[3] / name          # e.g. eval15/high/001.png
    lq = root / layout[2] / name          # e.g. eval15/low/001.png
    if not gt.exists():                   # extension fallback
        gt = next((root / layout[3] / (Path(name).stem + e) for e in (".png", ".jpg", ".jpeg")), None)
        lq = next((root / layout[2] / (Path(name).stem + e) for e in (".png", ".jpg", ".jpeg")), None)
    return lq, gt

n = min(3, len(preds))
fig, axes = plt.subplots(3, n, figsize=(5 * n, 12))
for i in range(n):
    name = preds[i].name
    lq, gt = find_pair(name)
    for row, (lbl, path) in enumerate([("input", lq), ("output", preds[i]), ("GT", gt)]):
        axes[row][i].imshow(np.asarray(Image.open(path).convert("RGB")))
        axes[row][i].set_title(f"{lbl}: {name}" if row == 0 else lbl)
        axes[row][i].axis("off")
plt.tight_layout(); plt.show()

## 6. (Optional) Cross-dataset generalisation — needs LIME/NPE/DICM attached

```python
cmd = [sys.executable, str(repo / "scripts" / "cross_dataset.py"),
       "--config", str(repo / CONFIG), "--ckpt", str(CKPT),
       "--input", "/kaggle/input/lime-images", "--name", "LIME",
       "--input", "/kaggle/input/npe-images",  "--name", "NPE",
       "--input", "/kaggle/input/dicom-images", "--name", "DICM",
       "--out", str(repo / "results" / RUN_NAME / "cross")]
subprocess.run(cmd, cwd=repo)
```

Requires `pip install pyiqa` (internet ON) for NIQE/BRISQUE/PI; the enhancement
step itself works without it.